# 02 — Bit allocation


In [ ]:
import sys
sys.path.insert(0, "..")
from autoquant import AutoQuantizer
import matplotlib.pyplot as plt
from collections import Counter

quantizer = AutoQuantizer("gpt2")
quantizer.analyze_sensitivity(num_samples=50)

targets = [0.1, 0.15, 0.2, 0.3]
results = []
for target in targets:
    r = quantizer.create_config(target)
    results.append(
        {"target": target, "expected": r["expected_size_gb"], "compression": r["compression_ratio"], "config": r["config"]}
    )
    print(f"Target {target:.2f} GB → {r['expected_size_gb']:.3f} GB ({r['compression_ratio']:.2f}x)")


In [ ]:
chosen = results[1]
assignments = chosen["config"]["bit_assignments"]
dist = Counter(assignments.values())

plt.figure(figsize=(8, 5))
labels = [f"INT{b}" if b < 16 else "FP16" for b in sorted(dist.keys())]
plt.bar(labels, [dist[b] for b in sorted(dist.keys())])
plt.title(f"Bit distribution @ target {chosen['target']:.2f} GB")
plt.ylabel("Layers")
plt.tight_layout()
plt.show()


In [ ]:
import json
with open("../config.json", "w", encoding="utf-8") as f:
    json.dump(chosen["config"], f, indent=2)
print("Saved ../config.json")
